# 04: Feature Extraction

## Objective
This notebook extracts features from aligned windows of VMO and VL EMG signals to create a structured dataset for subsequent machine-learning analysis.

## Workflow
1. Load the raw EMG data from Google Drive.
2. Divide both channels into 100-sample windows with a 50-sample step.
3. Rectify the signals and calculate RMS and integrated EMG (iEMG) ratios.
4. Calculate peak-time differences, onset-time differences, and cross-correlation features.
5. Assign preliminary VMO-dominant or VL-dominant labels based on the RMS ratio.
6. Save the extracted features to an Excel dataset.

## Output
A window-level feature dataset containing signal measurements, derived features, and preliminary activity labels.

## Interpretation note
The labels are generated using a simple RMS-ratio rule and should not be interpreted as clinically validated classifications.

In [ ]:
import numpy as np
import pandas as pd

# LOAD
from google.colab import drive

# Step 1: Mount Drive
drive.mount('/content/drive')

# Step 2: Load raw EMG data
file_path = "/content/drive/MyDrive/emg_final.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()

VMO = df.iloc[:, 0].values
VL  = df.iloc[:, 1].values

# PARAMETERS
window_size = 100
step_size = 50

def rms(signal):
    return np.sqrt(np.mean(signal**2))

def find_onset(signal):
    sorted_vals = np.sort(signal)
    cutoff = int(0.2 * len(sorted_vals))
    baseline = sorted_vals[:cutoff]

    mu = np.mean(baseline)
    sigma = np.std(baseline)
    threshold = mu + 3 * sigma

    for i in range(len(signal) - 5):
        if np.all(signal[i:i+5] > threshold):
            return i
    return None

rows = []

window_id = 0

for start in range(0, len(VMO) - window_size, step_size):
    end = start + window_size

    # SAME WINDOW → EVERYTHING BELOW USES THIS
    vmo_raw = VMO[start:end]
    vl_raw  = VL[start:end]

    vmo_rect = np.abs(vmo_raw)
    vl_rect  = np.abs(vl_raw)

    # FEATURES
    vmo_rms = rms(vmo_rect)
    vl_rms = rms(vl_rect)
    rms_ratio = vmo_rms / vl_rms if vl_rms != 0 else None

    vmo_iemg = np.sum(vmo_rect)
    vl_iemg = np.sum(vl_rect)
    iemg_ratio = vmo_iemg / vl_iemg if vl_iemg != 0 else None

    vmo_peak = np.argmax(vmo_rect)
    vl_peak = np.argmax(vl_rect)

    n = len(vmo_rect)
    peak_diff = (vmo_peak - vl_peak) / n

    vmo_onset = find_onset(vmo_rect)
    vl_onset = find_onset(vl_rect)

    onset_diff = None
    if vmo_onset is not None and vl_onset is not None:
        onset_diff = (vmo_onset - vl_onset) / n

    cross_corr = np.max(np.correlate(vmo_rect - np.mean(vmo_rect),
                                    vl_rect - np.mean(vl_rect),
                                    mode='full'))

    # STATUS
    status = "VMO_dominant" if rms_ratio and rms_ratio > 1 else "VL_dominant"

    # STORE ROW (ALL FROM SAME WINDOW)
    rows.append({
        "window_id": window_id,
        "start_index": start,
        "end_index": end,

        "VMO_RMS": vmo_rms,
        "VL_RMS": vl_rms,
        "RMS_ratio": rms_ratio,

        "iEMG_ratio": iemg_ratio,

        "peak_time_diff": peak_diff,
        "onset_diff": onset_diff,

        "cross_corr": cross_corr,

        "Status": status
    })

    window_id += 1

# SAVE
final_df = pd.DataFrame(rows)
final_df.to_excel("/content/drive/MyDrive/final_emg_dataset.xlsx", index=False)
print("Dataset created with aligned windows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Dataset created with aligned windows
